# Preprocessing Dataset Sentiment Analysis - Ulasan Aplikasi Gojek

Pipeline pembersihan dan alignment dataset dari 3 annotator untuk keperluan Aspect-Based Sentiment Analysis (ABSA).

## 1. Import Library

In [91]:
import pandas as pd
import numpy as np
import os
import re

print("Library berhasil diimport.")

Library berhasil diimport.


## 2. Load Dataset

In [92]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

RAW_DIR = "raw_dataset"

df1 = pd.read_csv(os.path.join(RAW_DIR, "dataset_annotator1.csv"))
# keep_default_na=False: agar string "None", "App", dll di Excel tidak otomatis jadi NaN
df2 = pd.read_excel(os.path.join(RAW_DIR, "dataset_annotator2.xlsx"), keep_default_na=False)
df3 = pd.read_excel(os.path.join(RAW_DIR, "dataset_annotator3.xlsx"), keep_default_na=False)

# Rename kolom sentence_id_raw -> sentence_id jika ada
for df in [df1, df2, df3]:
    if "sentence_id_raw" in df.columns:
        df.rename(columns={"sentence_id_raw": "sentence_id"}, inplace=True)

def _get_prefix(sid):
    m = re.match(r"^(review_\d+)", str(sid))
    return m.group(1) if m else str(sid)

# Forward-fill & back-fill category dan sentiment untuk annotator1 & 2
# Menangani merged cells di Excel: annotator sering menulis label hanya di 1 baris dalam grup
for _df in [df1, df2]:
    _df["_prefix"] = _df["sentence_id"].apply(_get_prefix)
    for col in ["category", "sentiment"]:
        if col in _df.columns:
            # Konversi empty string ke NaN agar ffill+bfill dapat berjalan
            _df[col] = _df[col].replace({"": np.nan, "nan": np.nan})
            _df[col] = _df.groupby("_prefix")[col].transform(lambda g: g.ffill().bfill())
    _df.drop(columns=["_prefix"], inplace=True)
# annotator3 TIDAK di-ffill: sel kosong = belum dilabelling, harus dibiarkan kosong

print("Dataset berhasil dimuat.")
print(f"Annotator1: {df1.shape}, Annotator2: {df2.shape}, Annotator3: {df3.shape}")
print(f"\nAnnotator2 - category NaN: {df2['category'].isna().sum()}, sentiment NaN: {df2['sentiment'].isna().sum()}")
print(f"Annotator3 - category kosong: {(df3['category']=='').sum() + df3['category'].isna().sum()}, sentiment kosong: {(df3['sentiment']=='').sum() + df3['sentiment'].isna().sum()}")

Dataset berhasil dimuat.
Annotator1: (38739, 5), Annotator2: (38789, 5), Annotator3: (38346, 5)

Annotator2 - category NaN: 1, sentiment NaN: 1
Annotator3 - category kosong: 18348, sentiment kosong: 18347


## 3. Inspect Dataset

In [93]:
def inspect_df(name, df):
    print(f"{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"Jumlah baris   : {len(df)}")
    print(f"Nama kolom     : {list(df.columns)}")
    print(f"\nTipe data:\n{df.dtypes}")
    print(f"\nMissing value:\n{df.isnull().sum()}")
    print()

inspect_df("Annotator 1", df1)
inspect_df("Annotator 2", df2)
inspect_df("Annotator 3", df3)

  Annotator 1
Jumlah baris   : 38739
Nama kolom     : ['sentence_id', 'at', 'content', 'category', 'sentiment']

Tipe data:
sentence_id    object
at             object
content        object
category       object
sentiment      object
dtype: object

Missing value:
sentence_id    0
at             0
content        0
category       0
sentiment      0
dtype: int64

  Annotator 2
Jumlah baris   : 38789
Nama kolom     : ['sentence_id', 'at', 'content', 'category', 'sentiment']

Tipe data:
sentence_id            object
at             datetime64[ns]
content                object
category               object
sentiment              object
dtype: object

Missing value:
sentence_id    0
at             0
content        0
category       1
sentiment      1
dtype: int64

  Annotator 3
Jumlah baris   : 38346
Nama kolom     : ['sentence_id', 'at', 'content', 'category', 'sentiment']

Tipe data:
sentence_id            object
at             datetime64[ns]
content                object
category            

## 4. Data Cleaning

In [94]:
EXPECTED_COLS = ["sentence_id", "at", "content", "category", "sentiment"]

# Pemetaan normalisasi label
CATEGORY_MAP = {
    "app and service": "app dan service",
    "app & service":   "app dan service",
    "service and app": "app dan service",
    "service & app":   "app dan service",
}
SENTIMENT_MAP = {
    "netral": "neutral",
}

def clean_df(df):
    df = df[[c for c in EXPECTED_COLS if c in df.columns]].copy()

    # Bersihkan content
    df["content"] = df["content"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    df = df[df["content"].notna() & (df["content"] != "") & (df["content"].str.lower() != "nan")]

    # Lowercase, strip, dan normalisasi label
    df["category"]  = df["category"].astype(str).str.strip().str.lower()
    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

    # Konversi "nan" (dari cell Excel kosong) → string kosong ""
    df["category"]  = df["category"].replace("nan", "")
    df["sentiment"] = df["sentiment"].replace("nan", "")

    # Normalisasi nilai label yang tidak konsisten
    df["category"]  = df["category"].replace(CATEGORY_MAP)
    df["sentiment"] = df["sentiment"].replace(SENTIMENT_MAP)

    df = df.reset_index(drop=True)
    return df

df1 = clean_df(df1)
df2 = clean_df(df2)
df3 = clean_df(df3)

print("Data cleaning selesai.")
print(f"Annotator1: {df1.shape}, Annotator2: {df2.shape}, Annotator3: {df3.shape}")

# Cek label unik setelah cleaning (sebelum alignment)
for name, df in [("Annotator1", df1), ("Annotator2", df2), ("Annotator3", df3)]:
    cats  = sorted(df["category"].unique())
    sents = sorted(df["sentiment"].unique())
    print(f"  {name} category={cats} | sentiment={sents}")

Data cleaning selesai.
Annotator1: (38739, 5), Annotator2: (38789, 5), Annotator3: (38346, 5)
  Annotator1 category=['app', 'app dan service', 'none', 'service'] | sentiment=['negative', 'neutral', 'positive']
  Annotator2 category=['', 'app', 'app dan service', 'none', 'service'] | sentiment=['', 'negative', 'neutral', 'positive']
  Annotator3 category=['', 'app', 'app dan service', 'none', 'service'] | sentiment=['', 'negative', 'neutral', 'positive']


## 5. Alignment Struktur Dataset

Annotator3 sebagai referensi struktur.
- **Prefix yang di annotator1/2 punya lebih banyak baris daripada di annotator3** → content digabung, label (category/sentiment) diambil dari annotator3.
- **Prefix yang jumlah barisnya sama (1:1)** → content, category, dan sentiment diambil dari annotator1/2 masing-masing.
- Label kosong pada annotator3 dibiarkan kosong (belum selesai dilabelling).

In [95]:
def extract_review_prefix(sentence_id):
    """Ekstrak prefix review dari sentence_id, misal review_2 dari review_2_1."""
    match = re.match(r"^(review_\d+)", str(sentence_id))
    return match.group(1) if match else str(sentence_id)

# Tambahkan kolom prefix ke masing-masing dataset
df1["prefix"] = df1["sentence_id"].apply(extract_review_prefix)
df2["prefix"] = df2["sentence_id"].apply(extract_review_prefix)
df3["prefix"] = df3["sentence_id"].apply(extract_review_prefix)

count1 = df1.groupby("prefix").size()
count2 = df2.groupby("prefix").size()
count3 = df3.groupby("prefix").size()

# Prefix yang perlu di-merge: sumber punya lebih banyak baris dari annotator3
merge_prefix1 = set(p for p in count3.index if count1.get(p, 0) > count3[p])
merge_prefix2 = set(p for p in count3.index if count2.get(p, 0) > count3[p])

# df3_ref: annotator3 tanpa kolom prefix, sebagai referensi struktur (sentence_id, at, content)
df3_ref = df3.copy()

print(f"Annotator1 - prefix yg perlu di-merge: {len(merge_prefix1)}")
print(f"Annotator2 - prefix yg perlu di-merge: {len(merge_prefix2)}")
print(f"Annotator3 referensi: {len(df3_ref)} baris")

Annotator1 - prefix yg perlu di-merge: 189
Annotator2 - prefix yg perlu di-merge: 197
Annotator3 referensi: 38346 baris


## 6. Alignment ke Struktur Annotator3

Untuk setiap baris di annotator3, tentukan konten dan label untuk annotator1/2:

In [96]:
def majority_label(series):
    """Ambil nilai terbanyak dari series, abaikan string kosong."""
    vals = series[(series != "") & series.notna()]
    return vals.mode()[0] if len(vals) > 0 else ""

def get_labels_from_source(df_source, df3_ref, merge_prefixes):
    """
    Petakan label (category, sentiment) dari df_source ke struktur df3_ref.

    - Content, sentence_id, at → selalu dari annotator3
    - Label untuk prefix 1:1 → ambil langsung dari source (exact sentence_id)
    - Label untuk prefix yang di-merge → majority vote semua baris source dengan prefix tsb
    - Prefix tidak ada di source → kosongkan label (fallback ke annotator3 seharusnya tidak perlu)
    """
    df_src = df_source.copy()
    # Pastikan prefix ada
    if "prefix" not in df_src.columns:
        df_src["prefix"] = df_src["sentence_id"].apply(extract_review_prefix)

    # Majority vote per prefix (digunakan untuk kasus merge)
    prefix_cat  = df_src.groupby("prefix")["category"].agg(majority_label).to_dict()
    prefix_sent = df_src.groupby("prefix")["sentiment"].agg(majority_label).to_dict()

    # Lookup exact per sentence_id (untuk kasus 1:1)
    src_by_sid = df_src.set_index("sentence_id")[["category", "sentiment"]].to_dict(orient="index")

    rows = []
    for _, ref_row in df3_ref.iterrows():
        sid    = ref_row["sentence_id"]
        prefix = ref_row["prefix"]

        # Content, at, sentence_id selalu dari annotator3
        content = ref_row.get("content", "")
        at_val  = ref_row.get("at", "")

        if prefix in merge_prefixes:
            # Baris di-merge → majority vote dari semua baris source untuk prefix ini
            category  = prefix_cat.get(prefix, "")
            sentiment = prefix_sent.get(prefix, "")
        elif sid in src_by_sid:
            # 1:1 → exact match
            category  = src_by_sid[sid].get("category", "")
            sentiment = src_by_sid[sid].get("sentiment", "")
        else:
            # sentence_id tidak ada di source (misal prefix beda)
            # coba pakai majority prefix sebagai fallback
            category  = prefix_cat.get(prefix, "")
            sentiment = prefix_sent.get(prefix, "")

        rows.append({
            "sentence_id": sid,
            "at":          at_val,
            "content":     content,
            "category":    category,
            "sentiment":   sentiment,
        })

    return pd.DataFrame(rows)

df1_aligned = get_labels_from_source(df1, df3_ref, merge_prefix1)
df2_aligned = get_labels_from_source(df2, df3_ref, merge_prefix2)
df3_aligned = df3_ref[["sentence_id", "at", "content", "category", "sentiment"]].copy().reset_index(drop=True)

print("Alignment selesai.")
print(f"Annotator1 aligned: {df1_aligned.shape}")
print(f"Annotator2 aligned: {df2_aligned.shape}")
print(f"Annotator3 aligned: {df3_aligned.shape}")

Alignment selesai.
Annotator1 aligned: (38346, 5)
Annotator2 aligned: (38346, 5)
Annotator3 aligned: (38346, 5)


## 7. Label Category dan Sentiment

Label `category` dan `sentiment` pada seluruh dataset menggunakan label dari annotator3 sebagai label final.

In [97]:
def check_labels(name, df):
    print(f"{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"category unik : {sorted(df['category'].unique())}")
    print(f"sentiment unik: {sorted(df['sentiment'].unique())}")
    ec = (df["category"]  == "").sum()
    es = (df["sentiment"] == "").sum()
    print(f"category kosong : {ec}")
    print(f"sentiment kosong: {es}")
    print()

check_labels("Annotator 1 (aligned)", df1_aligned)
check_labels("Annotator 2 (aligned)", df2_aligned)
check_labels("Annotator 3 (aligned)", df3_aligned)

  Annotator 1 (aligned)
category unik : ['', 'app', 'app dan service', 'none', 'service']
sentiment unik: ['', 'negative', 'neutral', 'positive']
category kosong : 16
sentiment kosong: 16

  Annotator 2 (aligned)
category unik : ['', 'app', 'app dan service', 'none', 'service']
sentiment unik: ['', 'negative', 'neutral', 'positive']
category kosong : 1
sentiment kosong: 1

  Annotator 3 (aligned)
category unik : ['', 'app', 'app dan service', 'none', 'service']
sentiment unik: ['', 'negative', 'neutral', 'positive']
category kosong : 18348
sentiment kosong: 18347



## 8. Validasi Dataset

In [98]:
def validate_df(name, df, ref_count):
    print(f"{'='*40}")
    print(f"  Validasi: {name}")
    print(f"{'='*40}")

    # Jumlah baris sesuai referensi
    row_ok = len(df) == ref_count
    print(f"Jumlah baris = {len(df)} (ref: {ref_count}) -> {'OK' if row_ok else 'GAGAL'}")

    # Tidak ada duplicate sentence_id
    dup = df["sentence_id"].duplicated().sum()
    print(f"Duplicate sentence_id: {dup} -> {'OK' if dup == 0 else 'GAGAL'}")

    # Content kosong (string kosong atau NaN) = belum dianotasi
    missing_content = (df["content"].isnull() | (df["content"].astype(str).str.strip() == "")).sum()
    print(f"Content kosong: {missing_content} -> {'OK' if missing_content == 0 else f'PERINGATAN ({missing_content} baris belum dianotasi)'}")
    print()

ref_count = len(df3_aligned)
validate_df("Annotator 1", df1_aligned, ref_count)
validate_df("Annotator 2", df2_aligned, ref_count)
validate_df("Annotator 3", df3_aligned, ref_count)

  Validasi: Annotator 1
Jumlah baris = 38346 (ref: 38346) -> OK
Duplicate sentence_id: 0 -> OK
Content kosong: 0 -> OK

  Validasi: Annotator 2
Jumlah baris = 38346 (ref: 38346) -> OK
Duplicate sentence_id: 0 -> OK
Content kosong: 0 -> OK

  Validasi: Annotator 3
Jumlah baris = 38346 (ref: 38346) -> OK
Duplicate sentence_id: 0 -> OK
Content kosong: 0 -> OK



## 9. Simpan Dataset

In [99]:
OUTPUT_DIR = "clean_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df1_aligned.to_csv(os.path.join(OUTPUT_DIR, "annotator1_clean.csv"), index=False, encoding="utf-8")
df2_aligned.to_csv(os.path.join(OUTPUT_DIR, "annotator2_clean.csv"), index=False, encoding="utf-8")
df3_aligned.to_csv(os.path.join(OUTPUT_DIR, "annotator3_clean.csv"), index=False, encoding="utf-8")

print(f"Dataset disimpan di folder '{OUTPUT_DIR}':")
print(f"  - annotator1_clean.csv")
print(f"  - annotator2_clean.csv")
print(f"  - annotator3_clean.csv")

Dataset disimpan di folder 'clean_dataset':
  - annotator1_clean.csv
  - annotator2_clean.csv
  - annotator3_clean.csv


## 10. Output Informasi Akhir

In [ ]:
def output_info(name, df):
    print(f"{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"Jumlah baris  : {len(df)}")
    missing_null = df.isnull().sum()
    missing_empty = (df == "").sum()
    missing_total = missing_null + missing_empty
    print(f"Missing value (NaN + string kosong):\n{missing_total}")
    print(f"\nPreview 5 baris pertama:")
    display(df.head())
    print()

output_info("Annotator 1 (Clean)", df1_aligned)
output_info("Annotator 2 (Clean)", df2_aligned)
output_info("Annotator 3 (Clean)", df3_aligned)

  Annotator 1 (Clean)
Jumlah baris  : 38346
Missing value (NaN + string kosong):
sentence_id     0
at              0
content         0
category       16
sentiment      16
dtype: int64

Preview 5 baris pertama:


,sentence_id,at,content,category,sentiment
0,review_1_1,2025-10-30 23:58:11,"Map suka eror,, driver dimana map dimana..biki...",app,negative
1,review_2_1,2025-10-30 23:42:51,terlalu mahal skrg.,service,negative
2,review_3_1,2025-10-30 23:34:57,bolehh lah,none,positive
3,review_4_1,2025-10-30 23:28:26,love you gojek....always the best😍,service,positive
4,review_5_1,2025-10-30 23:06:34,mantap gojek,service,positive



  Annotator 2 (Clean)
Jumlah baris  : 38346
Missing value (NaN + string kosong):
sentence_id    0
at             0
content        0
category       1
sentiment      1
dtype: int64

Preview 5 baris pertama:


,sentence_id,at,content,category,sentiment
0,review_1_1,2025-10-30 23:58:11,"Map suka eror,, driver dimana map dimana..biki...",app,negative
1,review_2_1,2025-10-30 23:42:51,terlalu mahal skrg.,app,negative
2,review_3_1,2025-10-30 23:34:57,bolehh lah,none,positive
3,review_4_1,2025-10-30 23:28:26,love you gojek....always the best😍,none,positive
4,review_5_1,2025-10-30 23:06:34,mantap gojek,none,positive



  Annotator 3 (Clean)
Jumlah baris  : 38346
Missing value (NaN + string kosong):
sentence_id        0
at                 0
content            0
category       18348
sentiment      18347
dtype: int64

Preview 5 baris pertama:


,sentence_id,at,content,category,sentiment
0,review_1_1,2025-10-30 23:58:11,"Map suka eror,, driver dimana map dimana..biki...",app dan service,negative
1,review_2_1,2025-10-30 23:42:51,terlalu mahal skrg.,app,negative
2,review_3_1,2025-10-30 23:34:57,bolehh lah,none,neutral
3,review_4_1,2025-10-30 23:28:26,love you gojek....always the best😍,service,positive
4,review_5_1,2025-10-30 23:06:34,mantap gojek,service,positive


: 